In [15]:
from sedona.spark import *
from pyspark.sql.functions import col, when, expr
#from sedona.sql.st_functions import ST_IsValid, ST_IsValidReason, ST_MakeValid DEPRECATED
from sedona.spark.sql.st_functions import ST_IsValid, ST_IsValidReason, ST_MakeValid
from pyspark.sql import DataFrame
import urllib.request
import json

In [16]:
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

26/03/02 19:11:24 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
26/03/02 19:11:24 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
26/03/02 19:11:24 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
26/03/02 19:11:24 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
26/03/02 19:11:24 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
26/03/02 19:11:24 WARN SimpleFunctionRegistry: The function rs_classify replaced a previously registered function.
26/03/02 19:11:24 WARN SimpleFunctionRegistry: The function rs_max_confidence replaced a previously registered function.
26/03/02 19:11:24 WARN SimpleFunctionRegistry: The function rs_segment replaced a previously registered fun

## Grab Area of Interest

In [17]:
import wkls

# seattle_dt = 'POLYGON ((-122.354307 47.612065, -122.318087 47.612065, -122.318087 47.626645, -122.354307 47.626645, -122.354307 47.612065))'

washington = wkls.us.wa.wkt()
seattle = wkls.us.wa.seattle.wkt()
kirkland = wkls.us.wa.kirkland.wkt()
bellevue = wkls.us.wa.bellevue.wkt()

# Small bounding box in central Kirkland (~0.5km²)
kirkland_small = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

In [18]:
SedonaKepler.create_map(sedona.sql(f"SELECT ST_GeomFromWKT('{kirkland_small}') AS geometry"), name="Area of Interest")

KeplerGl(data={'Area of Interest': {'index': [0], 'columns': ['geometry'], 'data': [['POLYGON ((-122.212000000…

## Grab residential Buildings

The Spatial Filter gets pushed down to storage level to only grab residential buildings that intersect the AOI

In [19]:
buildings_df = sedona.table("wherobots_open_data.overture_maps_foundation.buildings_building")\
                .filter(f"""
                    ST_Intersects(geometry, ST_GeomFromWKT('{kirkland_small}'))
                    AND subtype == 'residential'
                """)
buildings_df.createOrReplaceTempView("buildings")
buildings_df.show(5)
buildings_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+-----+-----------+--------+-----------------+-----+---------+--------------+----------+----------------------+----------+---------+------------+---------------+-------------+----------+--------------+----------------+----------+-----------+
|                  id|            geometry|                bbox|version|             sources|level|    subtype|   class|           height|names|has_parts|is_underground|num_floors|num_floors_underground|min_height|min_floor|facade_color|facade_material|roof_material|roof_shape|roof_direction|roof_orientation|roof_color|roof_height|
+--------------------+--------------------+--------------------+-------+--------------------+-----+-----------+--------+-----------------+-----+---------+--------------+----------+----------------------+----------+---------+------------+---------------+-------------+----------+--------------+----------------+----------+-----------

77

## Grab Overture Places

Same Spatial filter pushdown concept with Places of Interest (POI) dataset.

Here, we apply a buffer of 1000m to the AOI to make sure buildings near the edge of the AOI have coverage of POIs on all sides

In [20]:
places_df = sedona.table("wherobots_open_data.overture_maps_foundation.places_place")\
                .filter(f"""
                    ST_Intersects(
                        geometry, 
                        ST_Buffer(ST_GeomFromWKT('{kirkland_small}'), 1000, true)
                    )
                """)
places_df.createOrReplaceTempView("places")
places_df.show(5)
places_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+--------------+-----+--------------------+----------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|      basic_category|            taxonomy|        confidence|            websites|             socials|emails|        phones|brand|           addresses|operating_status|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+------+--------------+-----+--------------------+----------------+
|5c91d43e-8bf1-4d5...|POINT (-122.18843...|{-122.18844, -122...|      6|[{, meta, CDL

1353

### Coffee Shops

In [21]:
coffee_df = places_df.filter("categories.primary == 'coffee_shop'")
coffee_df.createOrReplaceTempView("coffee")
coffee_df.show(3)
coffee_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+------------------+--------------------+--------------------+------+----------------+--------------------+--------------------+----------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|basic_category|            taxonomy|        confidence|            websites|             socials|emails|          phones|               brand|           addresses|operating_status|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+------------------+--------------------+--------------------+------+----------------+--------------------+--------------------+----------------+
|51417ab7-fbf9-4ca...|POINT (-122.21029...|{-122.210

14

### Grocery Stores

In [22]:
grocery_df = places_df.filter("categories.primary == 'grocery_store'")
grocery_df.createOrReplaceTempView("grocery")
grocery_df.show(3)
grocery_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+------+----------------+-----+--------------------+----------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|basic_category|            taxonomy|         confidence|            websites|             socials|emails|          phones|brand|           addresses|operating_status|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+------+----------------+-----+--------------------+----------------+
|fbababa9-5e94-4b0...|POINT (-122.20401...|{-122.20402, -122...|      6|[{, meta, CDLA-Pe...|{

4

### Child Care

In [23]:
child_care_df = places_df.filter("categories.primary == 'child_care_and_day_care'")
child_care_df.createOrReplaceTempView("child_care")
child_care_df.show(3)
child_care_df.count()

+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+
| id|geometry|bbox|version|sources|names|categories|basic_category|taxonomy|confidence|websites|socials|emails|phones|brand|addresses|operating_status|
+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+
+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+



0

### Gyms

In [24]:
gym_df = places_df.filter("categories.primary == 'gym'")
gym_df.createOrReplaceTempView("gym")
gym_df.show(3)
gym_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+------------------+--------------------+--------------------+--------------------+----------------+-----+--------------------+----------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|basic_category|            taxonomy|        confidence|            websites|             socials|              emails|          phones|brand|           addresses|operating_status|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------+--------------------+------------------+--------------------+--------------------+--------------------+----------------+-----+--------------------+----------------+
|06f0b163-55c1-43e...|POINT (-122.20790...|{-122.20791,

17

### Hospitals

In [25]:
hospital_df = places_df.filter("categories.primary == 'hospital'")
hospital_df.createOrReplaceTempView("hospital")
hospital_df.show(3)
hospital_df.count()

+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+
| id|geometry|bbox|version|sources|names|categories|basic_category|taxonomy|confidence|websites|socials|emails|phones|brand|addresses|operating_status|
+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+
+---+--------+----+-------+-------+-----+----------+--------------+--------+----------+--------+-------+------+------+-----+---------+----------------+



0

### Transit Stops

In [26]:
transit_df = places_df.filter("categories.primary == 'transportation'")
transit_df.createOrReplaceTempView("transit")
transit_df.show(3)
transit_df.count()

+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+------+--------------+-----+--------------------+----------------+
|                  id|            geometry|                bbox|version|             sources|               names|          categories|      basic_category|            taxonomy|         confidence|            websites|             socials|emails|        phones|brand|           addresses|operating_status|
+--------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+------+--------------+-----+--------------------+----------------+
|76d7c48a-5259-4e1...|POINT (-122.19782...|{-122.19782, -122...|      6|[{, meta, 

2

### Schools

In [27]:
schools_df = sedona.table("org_catalog.gde_bronze.schools_bronze")\
                .filter(f"ST_Intersects(geometry, ST_GeomFromWKT('{bellevue}'))")
schools_df.createOrReplaceTempView("school")
schools_df.show(3)
schools_df.count()

+--------+-------+---------------------+------+-------+-------+-----+----------+----------+-------------+------------+-------+-------+-------------------+-----------+--------------+------+------+-----+---------+------+--------------+----------+-------------+
|geometry|AYPCode|CongressionalDistrict|County|ESDCode|ESDName|Email|GeoCoded_X|GeoCoded_Y|GradeCategory|HighestGrade|LEACode|LEAName|LegislativeDistrict|LowestGrade|MailingAddress|NCES_X|NCES_Y|Phone|Principal|School|SchoolCategory|SchoolCode|SingleAddress|
+--------+-------+---------------------+------+-------+-------+-----+----------+----------+-------------+------------+-------+-------+-------------------+-----------+--------------+------+------+-----+---------+------+--------------+----------+-------------+
+--------+-------+---------------------+------+-------+-------+-----+----------+----------+-------------+------------+-------+-------+-------------------+-----------+--------------+------+------+-----+---------+------+-----

0

## Perform KNN join between all essential amenities and residential buildings in downtown Seattle area

In [35]:
%%time

#Some datasets have cero ameties as bounding box is small, so need to use LEFT JOIN to avoid error, BUT apparently not supported with KNN

# Materialize buildings first
sedona.sql("CREATE OR REPLACE TEMP VIEW bldgs AS SELECT * FROM buildings")

# Run each KNN separately, skipping empty datasets entirely
sedona.sql('''CREATE OR REPLACE TEMP VIEW nearest_coffee AS
    SELECT b.id AS building_id, c.id AS coffee_id, c.names['primary'] AS coffee_name,
           c.geometry AS coffee_geometry, ST_DistanceSpheroid(b.geometry, c.geometry) AS coffee_distance
    FROM bldgs b JOIN coffee c ON ST_KNN(b.geometry, c.geometry, 1, true)''')

sedona.sql('''CREATE OR REPLACE TEMP VIEW nearest_grocery AS
    SELECT b.id AS building_id, g.id AS grocery_id, g.names['primary'] AS grocery_name,
           g.geometry AS grocery_geometry, ST_DistanceSpheroid(b.geometry, g.geometry) AS grocery_distance
    FROM bldgs b JOIN grocery g ON ST_KNN(b.geometry, g.geometry, 1, true)''')

sedona.sql('''CREATE OR REPLACE TEMP VIEW nearest_transit AS
    SELECT b.id AS building_id, t.id AS transit_id, t.names['primary'] AS transit_name,
           t.geometry AS transit_geometry, ST_DistanceSpheroid(b.geometry, t.geometry) AS transit_distance
    FROM bldgs b JOIN transit t ON ST_KNN(b.geometry, t.geometry, 1, true)''')

# Final join — LEFT JOIN the empty ones 
nearest_places_df = sedona.sql('''
    SELECT b.*,
        nc.coffee_id, nc.coffee_name, nc.coffee_geometry, nc.coffee_distance,
        ng.grocery_id, ng.grocery_name, ng.grocery_geometry, ng.grocery_distance,
        nt.transit_id, nt.transit_name, nt.transit_geometry, nt.transit_distance
    FROM bldgs b
    LEFT JOIN nearest_coffee nc ON b.id = nc.building_id
    LEFT JOIN nearest_grocery ng ON b.id = ng.building_id
    LEFT JOIN nearest_transit nt ON b.id = nt.building_id
''')

nearest_places_df.cache().count()

26/03/02 19:18:55 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.
26/03/02 19:18:55 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.
26/03/02 19:18:55 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.
                                                                                

CPU times: user 23 ms, sys: 1.82 ms, total: 24.8 ms
Wall time: 12.1 s


77

In [36]:
nearest_places_df.show(5)

+--------------------+--------------------+--------------------+-------+--------------------+-----+-----------+--------+-----------------+-----+---------+--------------+----------+----------------------+----------+---------+------------+---------------+-------------+----------+--------------+----------------+----------+-----------+--------------------+--------------------+--------------------+------------------+--------------------+------------+--------------------+------------------+--------------------+--------------------+--------------------+-----------------+
|                  id|            geometry|                bbox|version|             sources|level|    subtype|   class|           height|names|has_parts|is_underground|num_floors|num_floors_underground|min_height|min_floor|facade_color|facade_material|roof_material|roof_shape|roof_direction|roof_orientation|roof_color|roof_height|           coffee_id|         coffee_name|     coffee_geometry|   coffee_distance|          groc

In [37]:
# Create a new Havasu (Iceberg) database

database = 'gde_silver'

sedona.sql(f'CREATE DATABASE IF NOT EXISTS org_catalog.{database}')
database

'gde_silver'

In [38]:
%%time

nearest_places_df.writeTo(f"org_catalog.{database}.residential_buildings_amenites_silver").createOrReplace()

CPU times: user 7.48 ms, sys: 1.04 ms, total: 8.52 ms
Wall time: 3.43 s


In [39]:
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
+----------+--------------------+-----------+

